In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MBA-Hive-Registration") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.sparkContext.setLocalProperty("spark.scheduler.pool", "pool3")
print("SparkSession + Hive aktif")

SparkSession + Hive aktif


In [2]:
#Membuat database
spark.sql("CREATE DATABASE IF NOT EXISTS mba_db")
spark.sql("USE mba_db")
print("Database mba_db siap")
spark.sql("SHOW DATABASES").show()

Database mba_db siap
+---------+
|namespace|
+---------+
|  default|
|   mba_db|
+---------+



In [9]:
#Membuat tabel pada database mba_db
spark.sql("DROP TABLE IF EXISTS mba_db.basket_transactions")

spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS mba_db.basket_transactions (
        Transaction_ID  STRING,
        Discount_Applied INT,
        product           ARRAY<STRING>,
        discount_segment STRING
    )
    PARTITIONED BY (Season STRING)
    STORED AS PARQUET
    LOCATION 's3a://bigdata/gold/basket_transactions/'
    TBLPROPERTIES ('parquet.compression'='SNAPPY')
""")
print("Tabel basket_transactions dibuat")

spark.sql("MSCK REPAIR TABLE mba_db.basket_transactions")
print("Partisi dipulihkan (MSCK REPAIR)")

Tabel basket_transactions dibuat
Partisi dipulihkan (MSCK REPAIR)


In [10]:
print("=== Daftar Tabel di mba_db ===")
spark.sql("SHOW TABLES IN mba_db").show()

print("=== Partisi yang Terdaftar ===")
spark.sql("SHOW PARTITIONS mba_db.basket_transactions").show()

print("=== Preview Data via HiveQL ===")
spark.sql("""
    SELECT Season, COUNT(*) as total_basket
    FROM mba_db.basket_transactions
    GROUP BY Season
    ORDER BY Season
""").show()

=== Daftar Tabel di mba_db ===
+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|   mba_db|basket_transactions|      false|
+---------+-------------------+-----------+

=== Partisi yang Terdaftar ===
+-------------+
|    partition|
+-------------+
|  Season=Fall|
|Season=Spring|
|Season=Summer|
|Season=Winter|
+-------------+

=== Preview Data via HiveQL ===
+------+------------+
|Season|total_basket|
+------+------------+
|  Fall|      200244|
|Spring|      200416|
|Summer|      199419|
|Winter|      200191|
+------+------------+



In [13]:
print("=== Sample Basket per Musim ===")
spark.sql("""
    SELECT Transaction_ID, Season, discount_applied, product
    FROM mba_db.basket_transactions
    WHERE Season = 'Winter'
    LIMIT 5
""").show(truncate=False)

print("=== Statistik Jumlah Item per Basket ===")
spark.sql("""
    SELECT 
        Season,
        MIN(size(product)) AS min_product,
        MAX(size(product)) AS max_product,
        ROUND(AVG(size(product)), 2) AS avg_product,
        COUNT(*) AS total_basket
    FROM mba_db.basket_transactions
    GROUP BY Season
    ORDER BY Season
""").show()

print("Pipeline Data Engineer selesai! Data siap untuk FP-Growth.")

=== Sample Basket per Musim ===
+--------------+------+----------------+---------------------------------------------------------------------+
|Transaction_ID|Season|discount_applied|product                                                              |
+--------------+------+----------------+---------------------------------------------------------------------+
|1000000009    |Winter|1               |[Soap, Baby Wipes, Soda]                                             |
|1000000020    |Winter|0               |[Pancake Mix, Vacuum Cleaner]                                        |
|1000000024    |Winter|0               |[Sponges, Vacuum Cleaner, Iron]                                      |
|1000000054    |Winter|0               |[Olive Oil, Feminine Hygiene Products, Toothpaste, Soap, Bath Towels]|
|1000000062    |Winter|0               |[Beef, Bread, Dustpan]                                               |
+--------------+------+----------------+----------------------------------------